# Feature Engineering Pipeline
## Aquaculture Pond Identification — Zindi GeoAI Challenge

This notebook engineers robust features from the raw 144-column tabular data:
1. **Preprocessing**: Replace -9999 → NaN, compute valid-month count features
2. **Spectral Indices** (8 indices computed per valid time step)
3. **Temporal Aggregation** (10 statistics computed over valid months only — NaN-aware)
4. **ROS Augmentation**: Randomly mask 6–8 months during training, recompute features (10× augmentation)
5. **Save** engineered feature matrices for model training

In [1]:
import pandas as pd
import numpy as np
from scipy import stats
import re, os, pickle
from sklearn.impute import SimpleImputer
import warnings
warnings.filterwarnings('ignore')

DATA_DIR = '.'
OUT_DIR  = '.'
np.random.seed(42)

# ---- Column parsing --------------------------------------------------------
train_raw = pd.read_csv(f'{DATA_DIR}/Train.csv')
test_raw  = pd.read_csv(f'{DATA_DIR}/Test.csv')

FEAT_COLS = [c for c in train_raw.columns if c not in ('ID', 'label')]

def parse_col(col):
    m = re.match(r'^(.+)_(\d{2})$', col)
    return (m.group(1), int(m.group(2))) if m else (None, None)

BANDS     = sorted({parse_col(c)[0] for c in FEAT_COLS if parse_col(c)[0]})
TIMESTEPS = sorted({parse_col(c)[1] for c in FEAT_COLS if parse_col(c)[1] is not None})

OPT_BANDS = ['blue', 'green', 'nir', 'nira', 're1', 're2', 're3', 'red', 'swir1', 'swir2']
SAR_BANDS = ['VH', 'VV']

print(f'Bands ({len(BANDS)}): {BANDS}')
print(f'Time steps: {TIMESTEPS}')
print(f'Feature cols: {len(FEAT_COLS)}')

Bands (12): ['VH', 'VV', 'blue', 'green', 'nir', 'nira', 're1', 're2', 're3', 'red', 'swir1', 'swir2']
Time steps: [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12]
Feature cols: 144


## Step 1: Preprocessing — Replace -9999 with NaN & Valid Month Counts

In [2]:
def preprocess(df):
    """Replace -9999 with NaN and add validity count features."""
    df = df.copy()
    df[FEAT_COLS] = df[FEAT_COLS].replace(-9999, np.nan)
    # Count valid observations per sample per sensor type
    opt_cols = [f'{b}_{str(ts).zfill(2)}' for b in OPT_BANDS for ts in TIMESTEPS
                if f'{b}_{str(ts).zfill(2)}' in df.columns]
    sar_cols = [f'{b}_{str(ts).zfill(2)}' for b in SAR_BANDS for ts in TIMESTEPS
                if f'{b}_{str(ts).zfill(2)}' in df.columns]
    df['n_valid_optical'] = df[opt_cols].notna().sum(axis=1) // len(OPT_BANDS)
    df['n_valid_sar']     = df[sar_cols].notna().sum(axis=1) // len(SAR_BANDS)
    return df

train = preprocess(train_raw)
test  = preprocess(test_raw)

print('Train valid optical months:', train['n_valid_optical'].value_counts().sort_index().to_dict())
print('Test  valid optical months:', test['n_valid_optical'].value_counts().sort_index().to_dict())
print(f'\nMissing values in train feature cols: {train[FEAT_COLS].isna().sum().sum():,}')
print(f'Missing values in test  feature cols: {test[FEAT_COLS].isna().sum().sum():,}')

Train valid optical months: {12: 1821}
Test  valid optical months: {2: 8, 3: 94, 4: 364, 5: 311, 6: 253}

Missing values in train feature cols: 0
Missing values in test  feature cols: 89,756


## Step 2: Spectral Index Computation Per Time Step

| Index | Formula | Physical Meaning |
|-------|---------|------------------|
| NDWI | (green−nir)/(green+nir) | Open water presence |
| MNDWI | (green−swir1)/(green+swir1) | Better pond-edge separation |
| NDVI | (nir−red)/(nir+red) | Vegetation density (low for water) |
| EVI | 2.5×(nir−red)/(nir+6×red−7.5×blue+1) | Enhanced vegetation, aerosol-robust |
| LSWI | (nir−swir1)/(nir+swir1) | Leaf/surface water content |
| NDRE | (re1−red)/(re1+red) | Red-edge — pond vs mangrove discrimination |
| SAR_ratio | VH − VV (dB) | Surface roughness — specular for water |
| SAR_RVI | 4×VH/(VV+VH) | Radar Vegetation Index |

In [3]:
def safe_div(a, b):
    """Element-wise (a-b)/(a+b) with NaN propagation."""
    denom = a + b
    return np.where((denom == 0) | np.isnan(denom), np.nan, (a - b) / denom)

def compute_spectral_indices(df):
    """
    Add 8 spectral index columns per time step to df.
    Indices computed only where underlying bands are valid (NaN otherwise).
    """
    df = df.copy()
    for ts in TIMESTEPS:
        s = str(ts).zfill(2)
        g  = df.get(f'green_{s}', pd.Series(np.nan, index=df.index)).values
        n  = df.get(f'nir_{s}',   pd.Series(np.nan, index=df.index)).values
        r  = df.get(f'red_{s}',   pd.Series(np.nan, index=df.index)).values
        s1 = df.get(f'swir1_{s}', pd.Series(np.nan, index=df.index)).values
        bl = df.get(f'blue_{s}',  pd.Series(np.nan, index=df.index)).values
        r1 = df.get(f're1_{s}',   pd.Series(np.nan, index=df.index)).values
        vh = df.get(f'VH_{s}',    pd.Series(np.nan, index=df.index)).values
        vv = df.get(f'VV_{s}',    pd.Series(np.nan, index=df.index)).values

        df[f'NDWI_{s}']     = safe_div(g, n)
        df[f'MNDWI_{s}']    = safe_div(g, s1)
        df[f'NDVI_{s}']     = safe_div(n, r)
        # EVI: 2.5*(nir-red)/(nir + 6*red - 7.5*blue + 1)
        evi_denom = n + 6*r - 7.5*bl + 1
        df[f'EVI_{s}']      = np.where(np.isnan(evi_denom) | (evi_denom==0), np.nan, 2.5*(n-r)/evi_denom)
        df[f'LSWI_{s}']     = safe_div(n, s1)
        df[f'NDRE_{s}']     = safe_div(r1, r)
        # SAR ratio (dB space: just difference)
        df[f'SAR_ratio_{s}'] = vh - vv
        # SAR RVI: 4*VH / (VV + VH)
        sar_sum = vv + vh
        df[f'SAR_RVI_{s}'] = np.where(np.isnan(sar_sum) | (sar_sum==0), np.nan, 4*vh/sar_sum)
    return df

train = compute_spectral_indices(train)
test  = compute_spectral_indices(test)

# Show new index columns
new_idx_cols = [c for c in train.columns if any(c.startswith(idx) for idx in
               ['NDWI_','MNDWI_','NDVI_','EVI_','LSWI_','NDRE_','SAR_ratio_','SAR_RVI_'])]
print(f'Spectral index columns added: {len(new_idx_cols)}')
print(f'Example columns: {new_idx_cols[:8]}')

Spectral index columns added: 96
Example columns: ['NDWI_01', 'MNDWI_01', 'NDVI_01', 'EVI_01', 'LSWI_01', 'NDRE_01', 'SAR_ratio_01', 'SAR_RVI_01']


## Step 3: Temporal Aggregation (NaN-Aware)

For each band and derived index, compute 10 statistics **over valid months only** (ignoring NaN):
`mean, std, min, max, amplitude (max−min), p25, p75, skewness, linear_trend_slope, count_valid`

This yields: **(12 raw bands + 8 derived indices) × 10 stats = 200 aggregate features** + 2 validity counts = **202 total features**

In [4]:
def temporal_slope(arr):
    """Linear trend slope over valid time steps (NaN-safe)."""
    valid_mask = ~np.isnan(arr)
    if valid_mask.sum() < 2:
        return np.nan
    x = np.where(valid_mask)[0]
    y = arr[valid_mask]
    slope, *_ = np.polyfit(x, y, 1)
    return slope

def build_features(df, bands_list, ts_list):
    """
    For each band in bands_list, stack its time-step columns into a (N, T) array
    and compute 10 NaN-aware temporal statistics.
    Returns a feature DataFrame.
    """
    feature_rows = {i: {} for i in range(len(df))}
    
    for band in bands_list:
        # Gather columns for this band across all time steps
        cols = [f'{band}_{str(ts).zfill(2)}' for ts in ts_list
                if f'{band}_{str(ts).zfill(2)}' in df.columns]
        if not cols:
            continue
        arr = df[cols].values.astype(float)  # shape (N, T)
        
        for i, row in enumerate(arr):
            valid = row[~np.isnan(row)]
            n = len(valid)
            feature_rows[i][f'{band}__mean']  = np.nanmean(row) if n > 0 else np.nan
            feature_rows[i][f'{band}__std']   = np.nanstd(row)  if n > 1 else 0.0
            feature_rows[i][f'{band}__min']   = np.nanmin(row)  if n > 0 else np.nan
            feature_rows[i][f'{band}__max']   = np.nanmax(row)  if n > 0 else np.nan
            feature_rows[i][f'{band}__amp']   = (np.nanmax(row) - np.nanmin(row)) if n > 1 else 0.0
            feature_rows[i][f'{band}__p25']   = np.nanpercentile(row, 25) if n > 0 else np.nan
            feature_rows[i][f'{band}__p75']   = np.nanpercentile(row, 75) if n > 0 else np.nan
            feature_rows[i][f'{band}__skew']  = float(stats.skew(valid)) if n > 2 else 0.0
            feature_rows[i][f'{band}__slope'] = temporal_slope(row)
            feature_rows[i][f'{band}__nvalid']= n
    
    return pd.DataFrame.from_dict(feature_rows, orient='index')

# All bands to aggregate: raw + derived indices
DERIVED = ['NDWI', 'MNDWI', 'NDVI', 'EVI', 'LSWI', 'NDRE', 'SAR_ratio', 'SAR_RVI']
ALL_BANDS = BANDS + DERIVED

print(f'Building features for {len(ALL_BANDS)} band types ...')
train_feats = build_features(train, ALL_BANDS, TIMESTEPS)
test_feats  = build_features(test,  ALL_BANDS, TIMESTEPS)

# Add validity count features
train_feats['n_valid_optical'] = train['n_valid_optical'].values
train_feats['n_valid_sar']     = train['n_valid_sar'].values
test_feats['n_valid_optical']  = test['n_valid_optical'].values
test_feats['n_valid_sar']      = test['n_valid_sar'].values

print(f'Train feature matrix: {train_feats.shape}')
print(f'Test  feature matrix: {test_feats.shape}')
print(f'NaN count in train:   {train_feats.isna().sum().sum()}')
print(f'NaN count in test:    {test_feats.isna().sum().sum()}')

Building features for 20 band types ...
Train feature matrix: (1821, 202)
Test  feature matrix: (1030, 202)
NaN count in train:   0
NaN count in test:    0


## Step 4: Handle Remaining NaN (Imputation)

Any remaining NaN values after aggregation (e.g., when ALL months are masked for a band) are imputed with the **median of the training set**. The imputer is fitted on training data only and applied to test data (no leakage).

In [5]:
# Median imputation - fit on train, apply to both
imputer = SimpleImputer(strategy='median')
FEAT_NAMES = list(train_feats.columns)

train_X = imputer.fit_transform(train_feats)
train_X = pd.DataFrame(train_X, columns=FEAT_NAMES)
train_y = train['label'].values
train_ids = train['ID'].values

test_X  = imputer.transform(test_feats)
test_X  = pd.DataFrame(test_X, columns=FEAT_NAMES)
test_ids = test['ID'].values

print(f'Post-imputation NaN in train: {train_X.isna().sum().sum()}')
print(f'Post-imputation NaN in test:  {test_X.isna().sum().sum()}')
print(f'\nFinal feature dimensions:')
print(f'  Train X: {train_X.shape}  |  y: {train_y.shape}')
print(f'  Test  X: {test_X.shape}')

# Save imputer for reuse in model notebooks
with open(f'{OUT_DIR}/imputer.pkl', 'wb') as f:
    pickle.dump(imputer, f)
print('\nImputer saved to imputer.pkl')

Post-imputation NaN in train: 0
Post-imputation NaN in test:  0

Final feature dimensions:
  Train X: (1821, 202)  |  y: (1821,)
  Test  X: (1030, 202)

Imputer saved to imputer.pkl


## Step 5: ROS Augmentation (Random Observation Selection)

**Strategy**: During training, randomly mask 6–8 of the 12 months (set to NaN), recompute aggregate features. This forces the model to learn representations that are robust to partial temporal observations — directly simulating the 4–6-month test condition.

We generate **10 augmented copies** per training sample (10× augmentation) = ~18,210 augmented training samples + 1,821 originals.

In [6]:
def ros_augment_one(row_df, n_mask_min=6, n_mask_max=8, seed=None):
    """
    Given a single-row DataFrame with all raw band columns,
    randomly mask n_mask consecutive or random months, then
    recompute spectral indices and temporal aggregates.
    Returns a 1D feature vector.
    """
    rng = np.random.default_rng(seed)
    df = row_df.copy()
    
    n_mask = rng.integers(n_mask_min, n_mask_max + 1)
    masked_ts = rng.choice(TIMESTEPS, size=n_mask, replace=False)
    
    # Null out the chosen time steps
    for ts in masked_ts:
        ts_str = str(ts).zfill(2)
        band_cols = [f'{b}_{ts_str}' for b in BANDS if f'{b}_{ts_str}' in df.columns]
        df[band_cols] = np.nan
    
    # Recompute spectral indices on masked data
    df = compute_spectral_indices(df)
    # Rebuild temporal aggregates
    feats = build_features(df, ALL_BANDS, TIMESTEPS)
    # Add validity counts (recounted)
    opt_cols = [f'{b}_{str(ts).zfill(2)}' for b in OPT_BANDS for ts in TIMESTEPS
                if f'{b}_{str(ts).zfill(2)}' in df.columns]
    sar_cols = [f'{b}_{str(ts).zfill(2)}' for b in SAR_BANDS for ts in TIMESTEPS
                if f'{b}_{str(ts).zfill(2)}' in df.columns]
    feats['n_valid_optical'] = df[opt_cols].notna().sum(axis=1).values // len(OPT_BANDS)
    feats['n_valid_sar']     = df[sar_cols].notna().sum(axis=1).values // len(SAR_BANDS)
    return feats

def build_augmented_dataset(train_df, n_augments=10, seed=42):
    """Generate ROS-augmented training features."""
    rng = np.random.default_rng(seed)
    aug_X_list = []
    aug_y_list = []
    
    total = len(train_df) * n_augments
    print(f'Generating {total:,} augmented samples ({n_augments}× augmentation)...')
    
    for i, (_, row) in enumerate(train_df.iterrows()):
        row_df = pd.DataFrame([row], columns=train_df.columns)
        label  = int(row['label'])
        seeds  = rng.integers(0, 1_000_000, size=n_augments)
        for s in seeds:
            aug_feats = ros_augment_one(row_df, seed=int(s))
            aug_X_list.append(aug_feats.iloc[0].values)
            aug_y_list.append(label)
        if (i+1) % 200 == 0:
            print(f'  Processed {i+1}/{len(train_df)} samples...')
    
    aug_X = pd.DataFrame(aug_X_list, columns=FEAT_NAMES)
    aug_y = np.array(aug_y_list)
    return aug_X, aug_y

print('Starting ROS augmentation (this takes ~2–3 minutes)...')
aug_X_raw, aug_y = build_augmented_dataset(train, n_augments=10)

# Impute augmented data using the already-fitted imputer
aug_X = pd.DataFrame(imputer.transform(aug_X_raw), columns=FEAT_NAMES)
print(f'\nAugmented dataset: {aug_X.shape}')
print(f'Class balance in augmented set: {np.bincount(aug_y)}')

# Stack: original + augmented (for training only)
combined_X = pd.concat([train_X, aug_X], ignore_index=True)
combined_y = np.concatenate([train_y, aug_y])
print(f'\nCombined (original + augmented) dataset: {combined_X.shape}')

Starting ROS augmentation (this takes ~2–3 minutes)...
Generating 18,210 augmented samples (10× augmentation)...
  Processed 200/1821 samples...
  Processed 400/1821 samples...
  Processed 600/1821 samples...
  Processed 800/1821 samples...
  Processed 1000/1821 samples...
  Processed 1200/1821 samples...
  Processed 1400/1821 samples...
  Processed 1600/1821 samples...
  Processed 1800/1821 samples...

Augmented dataset: (18210, 202)
Class balance in augmented set: [10860  7350]

Combined (original + augmented) dataset: (20031, 202)


In [7]:
# Save all feature matrices for downstream notebooks
train_X.to_parquet(f'{OUT_DIR}/train_X_orig.parquet')
pd.DataFrame({'label': train_y}).to_parquet(f'{OUT_DIR}/train_y.parquet')
test_X.to_parquet(f'{OUT_DIR}/test_X.parquet')
combined_X.to_parquet(f'{OUT_DIR}/train_X_augmented.parquet')
pd.DataFrame({'label': combined_y}).to_parquet(f'{OUT_DIR}/train_y_augmented.parquet')

# Save IDs
pd.DataFrame({'ID': train_ids}).to_parquet(f'{OUT_DIR}/train_ids.parquet')
pd.DataFrame({'ID': test_ids}).to_parquet(f'{OUT_DIR}/test_ids.parquet')

# Save feature names
with open(f'{OUT_DIR}/feature_names.pkl', 'wb') as f:
    pickle.dump(FEAT_NAMES, f)

print('Saved:')
for fname in ['train_X_orig.parquet', 'train_y.parquet', 'test_X.parquet',
              'train_X_augmented.parquet', 'train_y_augmented.parquet',
              'train_ids.parquet', 'test_ids.parquet', 'feature_names.pkl', 'imputer.pkl']:
    size = os.path.getsize(f'{OUT_DIR}/{fname}') / 1024
    print(f'  {fname:35s}  {size:6.1f} KB')

Saved:
  train_X_orig.parquet                 2577.8 KB
  train_y.parquet                         1.8 KB
  test_X.parquet                       1523.0 KB
  train_X_augmented.parquet            23581.2 KB
  train_y_augmented.parquet               2.7 KB
  train_ids.parquet                      26.3 KB
  test_ids.parquet                       13.7 KB
  feature_names.pkl                       2.5 KB
  imputer.pkl                             4.6 KB


## Feature Engineering Summary

| Step | Output |
|------|--------|
| Preprocessing | -9999 → NaN; `n_valid_optical`, `n_valid_sar` features added |
| Spectral indices | 8 indices × 12 time steps = 96 intermediate columns |
| Temporal aggregation | (12 raw + 8 derived) × 10 stats = 200 aggregate features |
| Validity counts | +2 features (n_valid_optical, n_valid_sar) |
| **Total features** | **202 engineered features** |
| Imputation | Median imputer fitted on training data only (no leakage) |
| ROS augmentation | 10× augmentation = ~18,210 augmented + 1,821 original = ~20,031 training samples |

**Key design decisions:**
- Aggregation is **NaN-aware**: stats computed only over valid months, mimicking test-time partial observation
- Imputer fitted on **training data only**, applied to test (strict no-leakage)
- Augmented samples are imputed after augmentation using the same training-fitted imputer

**Next**: `03_models.ipynb` — stratified K-fold cross-validation with LightGBM, XGBoost, CatBoost + Optuna tuning